In [1]:
import pandas as pd


In [2]:
# full run AI cost
# Row 666/666 | in=2279626 out=239194 | invalid=0 | cumulative_est=$1.294561

In [4]:
retrievals = pd.read_csv('full_samples/retrievals.csv')

# Retrievals with no AIO text

In [5]:
retrievals['aio_text_char_count'] = retrievals['aio_text'].str.len()

In [6]:
retrievals['aio_text_char_count'].min()

np.float64(86.0)

In [7]:
retrievals[retrievals.aio_text.isna()]

,retrieval_id,aio_presence,aio_text,num_words,num_aio_sources,query_idx,query,category,subjectivity,answer_type,...,pct_Wikipedia,domain_category_entropy,log_population,News and Information Index,percent_min_bachelors,flesch_kincaid,citations_per_100_words,hedges_per_100_words,county_seat_to_state_ratio_normalized,aio_text_char_count
460,45b6b8e6-6e65-4237-abac-780eed86e333,1,NaN,0,NaN,80,why is COUNTYSEAT STATE so bad,safety,subjective,open-ended,...,NaN,NaN,12.380085,65.0,0.27741,NaN,NaN,NaN,NaN,NaN


# Sources with no snippets

In [8]:
aio_sources = pd.read_csv('full_samples/aio_sources.csv')

In [9]:
aio_sources[aio_sources.source_text.isna()]['retrieval_id'].unique()

<StringArray>
['b6fccd5b-6354-42dc-9935-cfe17756bc37',
 'ef258a83-de3c-4862-b328-d04631ad56a3',
 '6e6e5ec1-f5a5-4204-a936-327e9917ae98',
 '9ab18ebd-aa21-433b-9afb-bf7c4e11fed5',
 'f0c22090-c41e-4907-90eb-65e8b0f4d06f',
 '6c69a2ab-30b4-4aa3-b446-6cb7c9aca82e']
Length: 6, dtype: str

In [10]:
serps = pd.read_csv('full_samples/serps.csv')

In [11]:
serps[serps.serps_lede.isna()]['retrieval_id'].unique()

<StringArray>
['50b7be54-1787-42b6-ae79-edb94a100532',
 'cdf6bce2-91ee-400e-81ca-9ca4440412d1',
 'e2bcba9e-ee96-4fc0-ad9f-71b72db121b1',
 '7a52232b-cc6e-4b64-9bff-2a5f973941c4',
 '696eafbf-62dd-4ffb-8f64-f23b7ba94b9e',
 'ebd45e3a-fe44-495e-a91c-47e4512ef782',
 '27668c61-8a5f-4492-832c-66c8c41fca18',
 '706efed2-6b9b-44ad-bffe-044ebcdeeda7',
 '1b0be1e0-4dde-484b-a8d8-f8284afd9d4b',
 '065ce0d6-3d01-4fc8-8d70-0ab1f118392b',
 '9df25474-2c2e-4ed7-bc58-6ca26b8ba92e',
 'a2781576-36c1-448c-a856-a50af531d4df',
 '9ab18ebd-aa21-433b-9afb-bf7c4e11fed5',
 'a8018c2b-50f7-4a45-b09c-e97f436575d1',
 '60587760-359c-443f-8671-11f025c933b0',
 '5e2c8171-f2a0-4baa-ba42-336e730443ff']
Length: 16, dtype: str

In [36]:
n_serps = serps.groupby('retrieval_id')['serps_lede'].nunique().reset_index().sort_values(by = "serps_lede")

In [35]:
aio_sources.groupby('retrieval_id')['source_text'].nunique().reset_index().sort_values(by = "source_text")

,retrieval_id,source_text
289,6e6e5ec1-f5a5-4204-a936-327e9917ae98,1
539,d3b60a75-5846-42af-974f-99f693ed2f48,1
252,64b2a5de-8334-4057-b6c2-f056e682468d,1
544,d4c8b330-de72-4873-8d51-72cd10e4b7a8,1
347,88d47eb1-fd76-4ac8-bb99-997ea5d9ea3b,1
...,...,...
263,67dc9077-199c-417f-a43e-a68fdbe4e381,14
661,ff83d314-e6e7-4ef8-b0be-d97f92f65b63,15
160,3c9fa7f3-491a-43c8-8b09-18ba6fff9676,15
197,48d3ef76-b862-4d12-ae39-f0f1fb931c1b,16


# Systematically truncated text

In [4]:
truncated = pd.read_csv('full_samples/confirmed_truncated.csv')

In [5]:
truncated

,retrieval_id,confirmed_removal
0,7882e435-41b2-4bf2-99ca-45f10843ad24,True
1,3f212451-bf4e-4e09-b06d-1abe62b56592,True
2,e142401f-c198-45eb-a97a-17be095efbb4,True
3,696eafbf-62dd-4ffb-8f64-f23b7ba94b9e,True
4,da829ff0-5ac6-4836-9108-6b939bf70b22,True
5,be56cf57-b751-431b-8f11-df3f1e637bc4,True
6,8aaf71bc-086e-4f5d-8a5c-beef8539b818,True


## Produce csv of retrievals for experiment

In [39]:
retrievals = pd.read_csv('full_samples/retrievals.csv')
truncated = pd.read_csv('full_samples/confirmed_truncated.csv')
serps = pd.read_csv('full_samples/serps.csv')

In [42]:
n_serps = serps.groupby('retrieval_id')['serps_lede'].nunique().reset_index()

In [40]:
retrievals.shape

(666, 41)

In [41]:
retrievals = pd.merge(retrievals,
truncated,
how = "left",
on = "retrieval_id")

In [43]:
retrievals = pd.merge(retrievals,
n_serps,
how = "left",
on = "retrieval_id")

In [44]:
retrievals.shape

(666, 43)

In [45]:
n_missing_aio_text = len(retrievals.loc[retrievals.aio_text.isna()])
print(f"N missing AIO text: {n_missing_aio_text}")
print(f"% missing AIO text: {round(100*(n_missing_aio_text/len(retrievals)), 2)}%")

n_no_serp_source =  len(retrievals.loc[retrievals.serps_lede == 0])
print(f"N 0 valid SERP sources: {n_missing_aio_text}")
print(f"% 0 valid SERP sources: {round(100*(n_no_serp_source/len(retrievals)), 2)}%")

n_truncated_aio_text = len(retrievals.loc[retrievals.confirmed_removal == True])
print(f"N truncated AIO text: {n_truncated_aio_text}")
print(f"% truncated AIO text: {round(100*(n_truncated_aio_text/len(retrievals)), 2)}%")

N missing AIO text: 1
% missing AIO text: 0.15%
N 0 valid SERP sources: 1
% 0 valid SERP sources: 0.15%
N truncated AIO text: 7
% truncated AIO text: 1.05%


In [46]:
experiment_retrievals = retrievals.loc[(~retrievals.aio_text.isna()) & 
(retrievals.confirmed_removal != True) &
(retrievals.serps_lede > 0)]

In [47]:
experiment_retrievals.shape

(657, 43)

In [48]:
experiment_retrievals.to_csv('full_samples/experiment_retrievals.csv')